# Clase 7 — LOB Modeling Examples

**Objetivo:** Aplicar el pipeline de L6 con dos palancas nuevas: features temporales y comparación de modelos.

---

En L6 terminamos con **49.3% de accuracy** — por debajo del azar corregido (50.7%).  
Hoy usamos el mismo pipeline con dos cambios: mejores features y modelos más complejos.  
¿Es suficiente para batir al azar de forma consistente?

**Dos ejemplos:**
1. Predecir **dirección** del precio con features temporales
2. Predecir **fill probability** de una limit order

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')

plt.style.use('dark_background')
CYAN, GREEN, RED, AMBER, PURPLE, MUTED = '#22d3ee', '#4ade80', '#f87171', '#f59e0b', '#a78bfa', '#a1a1aa'

# Cargamos desde donde L6 dejó el trabajo
df = pd.read_csv('../06-lob-data-science-pipeline/data/lob_features.csv')
print(f'Datos cargados: {len(df)} snapshots, {len(df.columns)} columnas')

## Palanca 1 — Features temporales

En L6 usamos solo el **snapshot actual**. El LOB es un sistema dinámico — su historial reciente también informa.

Añadimos 3 features de memoria:
- `imbalance_mean_5`: media del imbalance en los últimos 5 snapshots (tendencia)
- `mid_momentum_5`: cuánto se ha movido el mid en los últimos 5 snapshots (velocidad)
- `imbalance_std_5`: variabilidad del imbalance reciente (régimen)

In [ ]:
df['imbalance_mean_5'] = df['imbalance'].rolling(5).mean()
df['mid_momentum_5']   = df['mid'] - df['mid'].shift(5)
df['imbalance_std_5']  = df['imbalance'].rolling(5).std()

# Visualización: raw vs rolling
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
N = 120

ax = axes[0]
ax.plot(df['imbalance'].values[:N], color=CYAN, alpha=0.4, linewidth=1, label='imbalance (t)')
ax.plot(df['imbalance_mean_5'].values[:N], color=AMBER, linewidth=2, label='imbalance_mean_5 (rolling)')
ax.axhline(0.5, color=MUTED, linewidth=0.8, linestyle='--')
ax.set_ylabel('imbalance', color=MUTED)
ax.set_title('Feature temporal: rolling mean suaviza el ruido y captura la tendencia', color='white')
ax.legend(labelcolor='white', loc='upper right')
ax.set_facecolor('#18181b')

ax = axes[1]
mom = df['mid_momentum_5'].values[:N]
colors_mom = [GREEN if v > 0 else RED for v in mom]
ax.bar(range(N), mom, color=colors_mom, alpha=0.7, width=1)
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set_ylabel('Δ mid × 5 (USD)', color=MUTED)
ax.set_xlabel('snapshot', color=MUTED)
ax.set_title('mid_momentum_5: velocidad del precio en los últimos 5 snapshots', color='white')
ax.set_facecolor('#18181b')

for ax in axes:
    ax.tick_params(colors=MUTED)
    for spine in ax.spines.values(): spine.set_edgecolor('#27272a')

fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

## Preparación: target y split

In [ ]:
# Target de dirección (misma definición que L6)
df['direction'] = (df['mid'].shift(-1) > df['mid']).astype(int)

# Target fill_in_3: ¿se ejecuta una limit bid en los próximos 3 snapshots?
fills = []
for i in range(len(df)):
    if i + 3 >= len(df): fills.append(np.nan); continue
    limit_p = df['bid_price_1'].iloc[i]
    fills.append(1 if any(df['mid'].iloc[i+1:i+4].values <= limit_p) else 0)
df['fill_in_3'] = fills

# Limpieza: eliminamos NaN de rolling + target
FEATS_L6       = ['imbalance', 'spread_pct']
FEATS_TEMPORAL = ['imbalance', 'spread_pct', 'wmid', 'depth_ratio',
                  'imbalance_mean_5', 'mid_momentum_5', 'imbalance_std_5']

df_clean = df.dropna(subset=FEATS_TEMPORAL + ['direction', 'fill_in_3']).reset_index(drop=True)
print(f'Filas limpias: {len(df_clean)} (vs 499 en L6 — perdemos {499-len(df_clean)} por rolling)')

split    = int(len(df_clean) * 0.7)
train_df = df_clean.iloc[:split].copy()
test_df  = df_clean.iloc[split:].copy()
print(f'Train: {len(train_df)}  |  Test: {len(test_df)}')

In [ ]:
# Referencia de L6: LR con 2 features
X_tr_l6 = train_df[FEATS_L6].values; y_tr = train_df['direction'].values
X_te_l6 = test_df[FEATS_L6].values;  y_te = test_df['direction'].values

lr_l6 = LogisticRegression(random_state=42, max_iter=1000).fit(X_tr_l6, y_tr)
acc_l6 = lr_l6.score(X_te_l6, y_te)
print(f'LR (L6 baseline, 2 features): {acc_l6:.1%}  ← punto de partida')

## Ejemplo 1 — Dirección con features temporales

In [ ]:
X_tr = train_df[FEATS_TEMPORAL].values
X_te = test_df[FEATS_TEMPORAL].values

def eval_clf(clf, name):
    clf.fit(X_tr, y_tr)
    tr = clf.score(X_tr, y_tr)
    te = clf.score(X_te, y_te)
    return {'name': name, 'train': tr, 'test': te, 'clf': clf}

lr_t  = eval_clf(LogisticRegression(random_state=42, max_iter=1000), 'LR temporal (7 feats)')
dt0   = eval_clf(DecisionTreeClassifier(random_state=42),            'DecisionTree sin podar')
dt2   = eval_clf(DecisionTreeClassifier(max_depth=2, random_state=42),'DecisionTree depth=2')
rf    = eval_clf(RandomForestClassifier(n_estimators=100, random_state=42), 'RandomForest')

print(f"{'Modelo':<28} {'Train':>8} {'Test':>8}")
print('-' * 46)
for r in [lr_t, dt0, dt2, rf]:
    marker = '  ← mejor test' if r['test'] == max(x['test'] for x in [lr_t,dt0,dt2,rf]) else ''
    print(f"{r['name']:<28} {r['train']:>8.1%} {r['test']:>8.1%}{marker}")

In [ ]:
# Visualización: train vs test para cada modelo — el overfitting es visible
models = [lr_t, dt0, dt2, rf]
names  = [m['name'].replace(' ', '\n') for m in models]
trains = [m['train'] for m in models]
tests  = [m['test']  for m in models]

x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4.5))
b1 = ax.bar(x - w/2, [t*100 for t in trains], w, label='Train', color=GREEN, alpha=0.8)
b2 = ax.bar(x + w/2, [t*100 for t in tests],  w, label='Test',  color=CYAN,  alpha=0.8)

ax.axhline(50.7, color=RED, linewidth=1.2, linestyle='--', alpha=0.7, label='Baseline L6 (50.7%)')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9, color='white')
ax.set_ylabel('Accuracy (%)', color=MUTED)
ax.set_ylim(40, 105)
ax.set_title('Train vs Test accuracy — la brecha grande = overfitting', color='white')
ax.legend(labelcolor='white')
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')

# Anotar el overfitting dramático
for b, v in zip(b1, trains):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.0%}',
            ha='center', color=GREEN, fontsize=9, fontweight='bold')
for b, v in zip(b2, tests):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.0%}',
            ha='center', color=CYAN, fontsize=9, fontweight='bold')

# Flecha al DT sin podar
ax.annotate('Memorizó el train.\nNo aprendió nada.',
            xy=(1 + w/2, tests[1]*100 + 0.5),
            xytext=(1.8, 65),
            color=RED, fontsize=9,
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

for spine in ax.spines.values(): spine.set_edgecolor('#27272a')
plt.tight_layout()
plt.show()

print('\nClave: el DT sin podar alcanza 100% en train y cae al 47% en test.')
print('El LR, con regularización implícita, generaliza mejor (55.5%).')

In [ ]:
# Curva de complejidad: cómo evoluciona train/test con la profundidad del árbol
depths  = [1, 2, 3, 4, 5, 6, 8, 10, 20]
tr_acc, te_acc = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr_acc.append(dt.score(X_tr, y_tr))
    te_acc.append(dt.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(depths, [v*100 for v in tr_acc], '-o', color=GREEN, linewidth=2, markersize=6, label='Train')
ax.plot(depths, [v*100 for v in te_acc], '-o', color=RED, linewidth=2, markersize=6, label='Test')
ax.fill_between(depths, [v*100 for v in tr_acc], [v*100 for v in te_acc],
                alpha=0.12, color=RED, label='Brecha (overfitting)')
ax.axhline(50.7, color=MUTED, linewidth=1, linestyle='--', alpha=0.6, label='Baseline L6')
ax.set_xlabel('max_depth del árbol', color=MUTED)
ax.set_ylabel('Accuracy (%)', color=MUTED)
ax.set_title('Curva de complejidad: a más profundidad, el árbol memoriza en vez de aprender', color='white')
ax.legend(labelcolor='white')
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
ax.tick_params(colors=MUTED)
for spine in ax.spines.values(): spine.set_edgecolor('#27272a')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances del RandomForest
imp_vals = rf['clf'].feature_importances_
imp_dict = dict(zip(FEATS_TEMPORAL, imp_vals))
sorted_imp = sorted(imp_dict.items(), key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = range(len(sorted_imp))
bars = ax.barh([f for f,_ in sorted_imp], [v for _,v in sorted_imp],
               color=CYAN, alpha=0.8)
ax.set_xlabel('Importancia relativa (RF)', color=MUTED)
ax.set_title('Feature importances — ninguno domina: el LOB tiene señal distribuida', color='white')
for bar, val in zip(bars, [v for _,v in sorted_imp]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', color=CYAN, fontsize=9)
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
ax.tick_params(colors=MUTED)
for spine in ax.spines.values(): spine.set_edgecolor('#27272a')
plt.tight_layout()
plt.show()

print('Las importancias están muy repartidas (~14% cada feature).')
print('Ninguna señal domina — esto es típico en mercados eficientes con datos pequeños.')

## Ejemplo 2 — Fill probability: ¿se ejecuta tu limit order?

En L5 colocaste una limit bid en el mejor nivel y observaste si se ejecutaba.  
Hoy lo convertimos en un problema de predicción.

**Target:** `fill_in_3 = 1` si el mid price baja hasta el nivel del bid en los próximos 3 snapshots.

**Intuición económica:** si hay muchos compradores (imbalance alto), el precio tiende a subir → tu bid no se ejecuta.  
Si hay vendedores (imbalance bajo), el precio cae → tu bid se ejecuta.

In [ ]:
y_tr_f = train_df['fill_in_3'].values
y_te_f = test_df['fill_in_3'].values

print(f'Fill rate (train): {y_tr_f.mean():.1%} de limit bids se ejecutan en 3 snapshots')
print(f'Fill rate (test):  {y_te_f.mean():.1%}')
print(f'Baseline always-majority: {max(y_te_f.mean(), 1-y_te_f.mean()):.1%}')

# Tasa empírica de fill por bucket de imbalance
df_clean['imb_bucket'] = pd.cut(df_clean['imbalance'],
                                 bins=[0, 0.35, 0.45, 0.55, 0.65, 1.0],
                                 labels=['< 0.35', '0.35–0.45', '0.45–0.55', '0.55–0.65', '> 0.65'])

fill_by_bucket = df_clean.groupby('imb_bucket', observed=True)['fill_in_3'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel izquierdo: fill rate por bucket
ax = axes[0]
colors_bucket = [CYAN if v < 0.55 else RED for v in fill_by_bucket]
bars = ax.bar(range(len(fill_by_bucket)), [v*100 for v in fill_by_bucket],
              color=colors_bucket, alpha=0.8)
ax.set_xticks(range(len(fill_by_bucket)))
ax.set_xticklabels(fill_by_bucket.index, fontsize=9, color='white')
ax.set_ylabel('Fill rate (%)', color=MUTED)
ax.set_title('Fill rate empírica por nivel de imbalance', color='white')
ax.set_ylim(0, 100)
for bar, v in zip(bars, fill_by_bucket):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{v:.0%}',
            ha='center', color='white', fontsize=10, fontweight='bold')
ax.set_facecolor('#18181b')

# Panel derecho: distribución de fills
ax = axes[1]
fill_pct = y_te_f.mean()
ax.pie([fill_pct, 1-fill_pct],
       labels=['Fill (1)', 'No fill (0)'],
       colors=[GREEN, RED], autopct='%1.0f%%',
       startangle=90, textprops={'color': 'white'})
ax.set_title('Distribución del target (test set)', color='white')
ax.set_facecolor('#18181b')

for ax in axes: ax.tick_params(colors=MUTED)
fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

print('\nObservación: imbalance bajo (muchos vendedores) → más fills. Imbalance alto → menos fills.')
print('La intuición económica se confirma empíricamente.')

In [ ]:
# Entrenamiento: RandomForest para fill probability
rf_fill = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fill.fit(X_tr, y_tr_f)

preds_fill = rf_fill.predict(X_te)
proba_fill = rf_fill.predict_proba(X_te)[:, 1]  # probabilidad de fill

acc_fill  = (preds_fill == y_te_f).mean()
acc_base  = max(y_te_f.mean(), 1-y_te_f.mean())

print(f'RF fill probability:')
print(f'  Train accuracy:          {rf_fill.score(X_tr,y_tr_f):.1%}')
print(f'  Test accuracy:           {acc_fill:.1%}')
print(f'  Baseline always-majority: {acc_base:.1%}')
print(f'  Mejora sobre baseline:   +{(acc_fill-acc_base)*100:.1f} pp')

In [ ]:
# Visualización: fill probability predicha vs imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: imbalance vs prob de fill
ax = axes[0]
c = [GREEN if f else RED for f in y_te_f]
ax.scatter(test_df['imbalance'], proba_fill, c=c, alpha=0.6, s=20)
ax.set_xlabel('imbalance', color=MUTED)
ax.set_ylabel('P(fill) predicha', color=MUTED)
ax.set_title('Fill probability predicha por el modelo', color='white')
ax.axhline(0.5, color=MUTED, linewidth=0.8, linestyle='--')
ax.set_facecolor('#18181b')
green_p = mpatches.Patch(color=GREEN, label='Fill real = 1')
red_p   = mpatches.Patch(color=RED,   label='Fill real = 0')
ax.legend(handles=[green_p, red_p], labelcolor='white')

# Aplicación: cuando usar limit vs market
ax = axes[1]
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]
imb_vals_ex = np.array([0.25, 0.4, 0.5, 0.6, 0.75])

# Predicción para cada nivel de imbalance
medians = {f: float(df_clean[f].median()) for f in FEATS_TEMPORAL}
results = []
for imb in imb_vals_ex:
    row = [[imb if f=='imbalance' else medians[f] for f in FEATS_TEMPORAL]]
    prob = float(rf_fill.predict_proba(row)[0][1])
    rec  = 'LIMIT' if prob > 0.5 else 'MARKET'
    results.append((imb, prob, rec))

ax.set_xlim(0, 1); ax.set_ylim(0, len(results)+1); ax.axis('off')
ax.set_facecolor('#18181b')
ax.set_title('Recomendación: ¿limit o market order?', color='white')

ax.text(0.05, len(results)+0.7, 'Imbalance', color=MUTED, fontsize=10, fontweight='bold')
ax.text(0.4,  len(results)+0.7, 'P(fill)', color=MUTED, fontsize=10, fontweight='bold')
ax.text(0.65, len(results)+0.7, 'Recomendación', color=MUTED, fontsize=10, fontweight='bold')

for i, (imb, prob, rec) in enumerate(reversed(results)):
    col = GREEN if rec == 'LIMIT' else AMBER
    ax.text(0.05, i+0.5, f'{imb:.2f}', color='white', fontsize=11, fontfamily='monospace')
    ax.text(0.4,  i+0.5, f'{prob:.2f}', color=CYAN, fontsize=11, fontfamily='monospace')
    ax.text(0.65, i+0.5, rec, color=col, fontsize=11, fontweight='bold')

fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

## Tabla resumen final

In [ ]:
print('=' * 60)
print(f'{"Modelo / Target":<30} {"Train":>8} {"Test":>8}')
print('=' * 60)

rows = [
    ('── Dirección del precio ──', '', ''),
    ('LR (L6 baseline, 2 feats)',   f'{lr_l6.score(X_tr_l6,y_tr):.1%}', f'{acc_l6:.1%}'),
    ('LR temporal (7 feats)',        f'{lr_t["train"]:.1%}', f'{lr_t["test"]:.1%}  ← mejor'),
    ('DecisionTree sin podar',       f'{dt0["train"]:.1%}', f'{dt0["test"]:.1%}  ← overfitting'),
    ('DecisionTree depth=2',         f'{dt2["train"]:.1%}', f'{dt2["test"]:.1%}'),
    ('RandomForest n=100',           f'{rf["train"]:.1%}',  f'{rf["test"]:.1%}'),
    ('── Fill probability (fill_in_3) ──', '', ''),
    ('Baseline always-majority',     '—', f'{acc_base:.1%}'),
    ('RandomForest n=100',           f'{rf_fill.score(X_tr,y_tr_f):.1%}', f'{acc_fill:.1%}  ← +{(acc_fill-acc_base)*100:.1f}pp'),
]

for name, tr, te in rows:
    if '──' in name:
        print(f'\n{name}')
    else:
        print(f'  {name:<28} {tr:>8} {te:>8}')

print('\n'+'=' * 60)
print('\nTakeaways:')
print('1. Features temporales mejoran la dirección: 49% → 55.5% (LR)')
print('2. El DT sin podar memoriza el train (100%) y falla en test (47%)')
print('3. Fill probability: señal causal más limpia → 54.8% (+2.7pp sobre baseline)')
print('4. Con 338 ejemplos, el modelo más simple (LR) generaliza mejor que RF')

## Limitaciones y puente a L8

**Lo que este pipeline no puede resolver:**
- 338 ejemplos de entrenamiento → varianza alta, señales inestables
- Sin costes de transacción ni market impact
- La señal del LOB (micro) no captura dinámicas de volumen a mayor escala

**Puente a L8:** Predecir dirección es una señal micro y ruidosa. Para ejecutar una orden grande (ejecutar 1000 BTC sin mover el mercado), necesitamos predecir **el perfil de volumen intradia** — una señal macro y más estable. Eso es VWAP, y es el tema de L8.